## 2D Correlation

In [1]:
%%writefile q10.cu
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define IMG_WIDTH  8
#define IMG_HEIGHT 8
#define LOW 10
#define HIGH 99
#define BLOCK_SIZE 8
#define MASK_WIDTH 3

// ── CUDA Kernel
__global__ void corr2DKernel(int *dN, int *dM, int *dP,
                              int imgWidth, int imgHeight, int maskWidth)
{
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;

    if (col < imgWidth && row < imgHeight) {
        int dPvalue = 0;
        for (int r = 0; r < maskWidth; r++) {
            for (int c = 0; c < maskWidth; c++) {
                int N_row = row + r;
                int N_col = col + c;
                if (N_row < imgHeight && N_col < imgWidth)
                    dPvalue += dN[N_row * imgWidth + N_col] * dM[r * maskWidth + c];
            }
        }
        dP[row * imgWidth + col] = dPvalue;
    }
}

// ── Host Function
__host__ void corr2D(int *hN, int *hM, int *hP,
                     int imgWidth, int imgHeight, int maskWidth)
{
    int *dN, *dM, *dP;
    int sizeN = imgWidth * imgHeight * sizeof(int);
    int sizeM = maskWidth * maskWidth * sizeof(int);

    cudaMalloc((void**)&dN, sizeN);
    cudaMalloc((void**)&dM, sizeM);
    cudaMalloc((void**)&dP, sizeN);

    cudaMemcpy(dN, hN, sizeN, cudaMemcpyHostToDevice);
    cudaMemcpy(dM, hM, sizeM, cudaMemcpyHostToDevice);

    dim3 DimBlock(BLOCK_SIZE, BLOCK_SIZE, 1);
    dim3 DimGrid((int)ceil((float)imgWidth  / BLOCK_SIZE),
                 (int)ceil((float)imgHeight / BLOCK_SIZE), 1);

    corr2DKernel<<<DimGrid, DimBlock>>>(dN, dM, dP, imgWidth, imgHeight, maskWidth);

    cudaMemcpy(hP, dP, sizeN, cudaMemcpyDeviceToHost);

    cudaFree(dN); cudaFree(dM); cudaFree(dP);
}

// ── Main
int main()
{
    int hN[IMG_HEIGHT][IMG_WIDTH];
    int hP[IMG_HEIGHT][IMG_WIDTH];
    int hM[MASK_WIDTH][MASK_WIDTH] = {{1,2,1},{2,4,2},{1,2,1}};

    srand(time(NULL));
    for (int r = 0; r < IMG_HEIGHT; r++)
        for (int c = 0; c < IMG_WIDTH; c++)
            hN[r][c] = rand() % (HIGH - LOW + 1) + LOW;

    printf("hN (Image %dx%d):\n", IMG_HEIGHT, IMG_WIDTH);
    for (int r = 0; r < IMG_HEIGHT; r++) {
        for (int c = 0; c < IMG_WIDTH; c++) printf("%4d", hN[r][c]);
        printf("\n");
    }

    printf("\nhM (Mask %dx%d):\n", MASK_WIDTH, MASK_WIDTH);
    for (int r = 0; r < MASK_WIDTH; r++) {
        for (int c = 0; c < MASK_WIDTH; c++) printf("%4d", hM[r][c]);
        printf("\n");
    }

    corr2D((int*)hN, (int*)hM, (int*)hP, IMG_WIDTH, IMG_HEIGHT, MASK_WIDTH);

    printf("\nhP (2D Correlation Result):\n");
    for (int r = 0; r < IMG_HEIGHT; r++) {
        for (int c = 0; c < IMG_WIDTH; c++) printf("%6d", hP[r][c]);
        printf("\n");
    }

    return 0;
}

Writing q10.cu


In [2]:
!nvcc -arch=sm_75 q10.cu -o q10

!./q10

hN (Image 8x8):
  34  35  67  85  25  57  77  19
  20  55  12  37  24  70  16  26
  26  32  34  97  78  19  57  55
  61  92  22  88  33  19  93  19
  44  60  94  60  69  33  31  79
  41  85  16  55  55  74  33  71
  96  19  31  36  81  40  44  42
  84  18  83  18  28  76  79  24

hM (Mask 3x3):
   1   2   1
   2   4   2
   1   2   1

hP (2D Correlation Result):
   579   683   788   774   749   674   418   126
   657   734   953   872   690   728   533   155
   916   953  1051   849   667   810   618   172
  1010  1012   978   874   732   783   684   248
   877   769   829   947   843   766   667   263
   760   608   696   865   905   809   558   179
   571   521   478   538   723   686   382    90
   203   202   147   150   259   258   127    24
